## The Train/Val/Test Split

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. Load the data
df = pd.read_csv('../data/Train.csv') # Update filename if necessary

# Clean up column names to be uniform (lowercase, replace spaces with underscores)
df.columns = df.columns.str.lower().str.replace('.', '_').str.replace(' ', '_')

# 2. Perform the split (60% Train, 20% Val, 20% Test)
# First, split off the 20% for the test set
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# Next, split the remaining 80% to get a 20% validation set
# (20% is exactly 1/4th of 80%, so we use test_size=0.25)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

# Reset indices for cleanliness
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# 3. Isolate the target variable ('reached_on_time_y_n')
y_train = df_train.reached_on_time_y_n.values
y_val = df_val.reached_on_time_y_n.values
y_test = df_test.reached_on_time_y_n.values

# Remove the target variable from the dataframes so we don't accidentally train on it
del df_train['reached_on_time_y_n']
del df_val['reached_on_time_y_n']
del df_test['reached_on_time_y_n']

print(f"Train size: {len(df_train)}")
print(f"Validation size: {len(df_val)}")
print(f"Test size: {len(df_test)}")

Train size: 6599
Validation size: 2200
Test size: 2200


## Exploratory Data Analysis

1. Missing Values: Are there any missing values in the dataset?

2. Target Distribution: What is the ratio of packages that arrive on time vs. late? (Use value_counts(normalize=True)).

3. Feature Importance (Categorical): Which warehouse block has the highest rate of late deliveries?

4. Feature Importance (Numerical): Is there a correlation between discount_offered and whether the package is late? (Hint: calculate the mean discount for late vs. on-time packages).

In [5]:
# Create a temporary dataframe for EDA so we don't mess up our clean training data
df_eda = df_train.copy()
df_eda['late_delivery'] = y_train 
# Note: 1 = Late (Did NOT reach on time), 0 = On Time (Reached on time)

# --- 1. Missing Values ---
print("--- Missing Values ---")
print(df_eda.isnull().sum())
print("\n")

# --- 2. Target Distribution ---
print("--- Target Distribution (Late vs On Time) ---")
# normalize=True gives us the percentage instead of the raw count
print(df_eda['late_delivery'].value_counts(normalize=True))
print("\n")

# --- 3. Feature Importance (Categorical: Warehouse) ---
print("--- Late Delivery Rate by Warehouse Block ---")
# Grouping by warehouse and calculating the mean of 'late_delivery' 
# gives us the exact percentage of packages that were late from each block.
print(df_eda.groupby('warehouse_block')['late_delivery'].mean().sort_values(ascending=False))
print("\n")

# --- 4. Feature Importance (Numerical: Discount) ---
print("--- Average Discount Offered (On Time vs Late) ---")
print(df_eda.groupby('late_delivery')['discount_offered'].mean())

--- Missing Values ---
id                     0
warehouse_block        0
mode_of_shipment       0
customer_care_calls    0
customer_rating        0
cost_of_the_product    0
prior_purchases        0
product_importance     0
gender                 0
discount_offered       0
weight_in_gms          0
late_delivery          0
dtype: int64


--- Target Distribution (Late vs On Time) ---
late_delivery
1    0.59903
0    0.40097
Name: proportion, dtype: float64


--- Late Delivery Rate by Warehouse Block ---
warehouse_block
D    0.606572
C    0.604244
B    0.603116
F    0.600915
A    0.578616
Name: late_delivery, dtype: float64


--- Average Discount Offered (On Time vs Late) ---
late_delivery
0     5.487906
1    18.850746
Name: discount_offered, dtype: float64


Look closely at that last output regarding the Average Discount Offered:

Packages that arrive on time: ~5.49 discount.

Packages that arrive late: ~18.85 discount.

This is a massive signal. It tells us that discount_offered is going to be a very strong predictor for our model. (From a business perspective, this might mean the company is slapping high discounts on packages they already know are going to be delayed, or perhaps highly discounted items are shipped via a slower, lower-priority tier!).

Since your data is clean (zero missing values) and you have identified some great features, it is time to build your Phase 1 Baseline Model.

Step 4: Baseline Model Training
In the Machine Learning Zoomcamp, we use Scikit-Learn's DictVectorizer. It is a fantastic tool because it automatically looks at your dataframe, leaves the numerical columns alone, and automatically applies One-Hot Encoding to all the categorical (text) columns.

What is happening here?
DictVectorizer: By converting our dataframe to a dictionary, DictVectorizer sees a column like warehouse_block: 'D' and turns it into warehouse_block=D: 1 and warehouse_block=A: 0, doing all the mathematical formatting for you.

predict vs predict_proba: Accuracy simply checks if the model guessed 1 or 0 correctly. ROC AUC checks how confident the model was. To calculate ROC AUC, we must use predict_proba to get the raw probability (e.g., "I am 82% sure this will be late").

In [6]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore') # Ignores some basic Scikit-Learn warnings

# 1. Convert DataFrames to lists of dictionaries (Required for DictVectorizer)
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

# 2. Fit the DictVectorizer and transform the data
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts) # Notice we ONLY transform the validation set, we don't fit!

# 3. Train the Baseline Logistic Regression Model
# (Using liblinear as it is a solid default solver for standard datasets)
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# 4. Make Predictions on the Validation Set
# For Accuracy, we need exact 1s and 0s
y_pred_bin = model.predict(X_val)

# For ROC AUC, we need the actual probability percentages of class 1 (Late)
y_pred_proba = model.predict_proba(X_val)[:, 1] 

# 5. Evaluate and Print Scores
accuracy = accuracy_score(y_val, y_pred_bin)
roc_auc = roc_auc_score(y_val, y_pred_proba)

print(f"Baseline Validation Accuracy: {accuracy:.4f}")
print(f"Baseline Validation ROC AUC:  {roc_auc:.4f}")

Baseline Validation Accuracy: 0.6477
Baseline Validation ROC AUC:  0.7218


## Phase 2: Complex Modeling

Logistic Regression is a linear model. It struggles to understand complex conditional rules. However, in this phase, we are going to use Tree-Based Ensembles (Random Forest and XGBoost). These models learn by creating a series of "If/Then" questions (e.g., "IF discount > 10 AND warehouse = D, THEN package is late"), which makes them incredibly powerful for tabular data like ours.

Here is your action plan for Phase 2. Paste this code into your next Jupyter Notebook cell. We are going to train a Random Forest and an XGBoost model right out of the box to see if they can beat your 0.7218 baseline.

In [7]:
!pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   -- ------------------------------------- 7.1/101.7 MB 33.0 MB/s eta 0:00:03
   ------- -------------------------------- 18.9/101.7 MB 43.7 MB/s eta 0:00:02
   ---------- ----------------------------- 27.0/101.7 MB 42.0 MB/s eta 0:00:02
   -------------- ------------------------- 35.7/101.7 MB 41.7 MB/s eta 0:00:02
   ---------------- ----------------------- 42.2/101.7 MB 39.5 MB/s eta 0:00:02
   -------------------- ------------------- 52.2/101.7 MB 40.4 MB/s eta 0:00:02
   ------------------------ --------------- 61.9/101.7 MB 41.4 MB/s eta 0:00:01
   ---------------------------- ----------- 72.4/101.7 MB 42.6 MB/s eta 0:00:01
   ------------------------------- -------- 79.7/101.7 MB 41.7 MB/s eta 0:00:01
   ---------------------------------- ----- 87.6/101.7 MB 41.2 MB/s eta 0:00:01
   ------------------------------------- -- 95.2/101.7 MB 4

In [8]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

# --- Model 1: Random Forest ---
# n_estimators is the number of trees, max_depth prevents the trees from getting too complex and overfitting
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict_proba(X_val)[:, 1]
roc_auc_rf = roc_auc_score(y_val, y_pred_rf)
print(f"Random Forest Validation ROC AUC: {roc_auc_rf:.4f}")


# --- Model 2: XGBoost ---
# XGBoost builds trees sequentially, learning from the mistakes of the previous trees.
xgb_model = XGBClassifier(
    n_estimators=100, 
    max_depth=5, 
    learning_rate=0.1, 
    random_state=42,
    use_label_encoder=False,
    eval_metric='auc'
)
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict_proba(X_val)[:, 1]
roc_auc_xgb = roc_auc_score(y_val, y_pred_xgb)
print(f"XGBoost Validation ROC AUC:       {roc_auc_xgb:.4f}")

Random Forest Validation ROC AUC: 0.7224
XGBoost Validation ROC AUC:       0.7309


In [9]:
import pickle

# The standard Zoomcamp naming convention for the exported model
output_file = 'xgboost_model.bin' 

# Open the file in 'wb' (write binary) mode
with open(output_file, 'wb') as f_out:
    # We save both the DictVectorizer (dv) and the XGBoost model (xgb_model) as a tuple
    pickle.dump((dv, xgb_model), f_out)

print(f"Success! Model and DictVectorizer saved to {output_file}")

Success! Model and DictVectorizer saved to xgboost_model.bin
